# Residential Electricity Short-Term Load Forecasting
## correctReleases — Clean Pipeline
**Author:** Chaitra R | **Enphase Energy** | **May 2026**

---
### How to use
1. Set `RUN_MODE` in Cell 1 to `'1year'` or `'2year'`
2. Run all cells top to bottom
3. Results saved to `run1_1year/` or `run2_2year/` automatically

| Run | Training data | Test data | Purpose |
|-----|--------------|-----------|----------|
| `1year` | 2018 Jan–Sep | 2018 Oct–Dec | Baseline experiment |
| `2year` | 2012 full + 2018 Jan–Sep | 2018 Oct–Dec | Prophet improvement |

## Cell 1: Configuration
> **Change `RUN_MODE` here to switch between experiments**

In [41]:
# ╔══════════════════════════════════════════════════════╗
# ║           SET RUN MODE HERE                         ║
# ╚══════════════════════════════════════════════════════╝
RUN_MODE = '2year'   # OPTIONS: '1year'  or  '2year'

# ── Paths ────────────────────────────────────────────────
import os
from pathlib import Path

BASE_DIR   = Path('.')                          # correctReleases/
DATA_DIR   = BASE_DIR / 'data'
RUN_DIR    = BASE_DIR / f'run1_1year_r2' if RUN_MODE == '1year' else BASE_DIR / 'run2_2year_r2'

# Data paths — always release_1.1
TS_2018    = DATA_DIR / 'timeseries_2018'
TS_2012    = DATA_DIR / 'timeseries_2012'
WX_2018    = DATA_DIR / 'weather_2018'
WX_2012    = DATA_DIR / 'weather_2012'
METADATA   = DATA_DIR / 'metadata' / 'CA_upgrade0.parquet'
SELECTED   = BASE_DIR / 'selected_buildings.csv'

# Output paths
PROCESSED  = RUN_DIR / 'processed'
PRED_DIR   = RUN_DIR / 'predictions'
MODEL_DIR  = RUN_DIR / 'models'
RESULTS    = RUN_DIR / 'results'

# Create all output folders
for d in [PROCESSED, PRED_DIR, MODEL_DIR, RESULTS,
          RESULTS/'xai'/'shap', RESULTS/'xai'/'lime',
          RESULTS/'xai'/'anchors', RESULTS/'xai'/'plots',
          RESULTS/'evaluation'/'plots']:
    Path(d).mkdir(parents=True, exist_ok=True)

# Train/test split
TEST_CUTOFF = '2018-10-01'

print(f'RUN MODE  : {RUN_MODE}')
print(f'RUN DIR   : {RUN_DIR}')
print(f'Train     : {"2012 full + 2018 Jan-Sep" if RUN_MODE == "2year" else "2018 Jan-Sep"}')
print(f'Test      : 2018 Oct-Dec')

RUN MODE  : 2year
RUN DIR   : run2_2year_r2
Train     : 2012 full + 2018 Jan-Sep
Test      : 2018 Oct-Dec


## Cell 2: Imports

In [42]:
import pandas as pd
import numpy as np
import pickle
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    r2_score, mean_squared_log_error
)
from sklearn.ensemble import GradientBoostingRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
print('✓ All imports successful')

✓ All imports successful


## Cell 3: Setup — Copy data from old folder
> Run this once to copy files from `resstock_data/` to `correctReleases/data/`
> Skip if already copied.

In [43]:
import shutil

# ── FIXED: Cell 6 — Data source check ───────────────────────────
# Previously this cell copied from '../resstock_data' (a folder that
# only existed on the original author's machine). Now that
# downloadFiles.py saves directly into ./data/timeseries_2018,
# ./data/weather_2018, etc., this cell just verifies the expected
# files are present and tells you clearly if something is missing
# instead of silently failing on every "Source not found" line.

required_dirs = {
    'timeseries_2018': TS_2018,
    'weather_2018'   : WX_2018,
}
# Only require 2012 data if this run actually needs it
if RUN_MODE == '2year':
    required_dirs['timeseries_2012'] = TS_2012
    required_dirs['weather_2012']    = WX_2012

print('Checking data availability...')
all_ok = True
for name, path in required_dirs.items():
    if not path.exists():
        print(f'  [MISSING] {name}: {path} does not exist')
        all_ok = False
        continue
    n_files = len(list(path.glob('*')))
    if n_files == 0:
        print(f'  [EMPTY]   {name}: {path} exists but has 0 files')
        all_ok = False
    else:
        print(f'  [OK]      {name}: {n_files} files in {path}')

if not METADATA.exists():
    print(f'  [MISSING] metadata: {METADATA} does not exist')
    all_ok = False
else:
    print(f'  [OK]      metadata: {METADATA}')

if not SELECTED.exists():
    print(f'  [MISSING] selected_buildings.csv: {SELECTED} does not exist')
    all_ok = False
else:
    print(f'  [OK]      selected_buildings.csv: {SELECTED}')

if all_ok:
    print('\n[OK] All required data is present. Ready to proceed.')
else:
    print('\n[ERROR] Some required data is missing.')
    print('  Run downloadFiles.py first to populate ./data/ with the')
    print('  required timeseries, weather, and metadata files.')


  ✓ Already exists: timeseries_2018
  ✓ Already exists: timeseries_2012
  ✓ Already exists: weather_2018
  ✓ Already exists: weather_2012
  ✓ Already exists: CA_upgrade0.parquet
  ✓ Already exists: selected_buildings.csv
  ✓ Already exists: available_files_2018.txt
  ✓ Already exists: available_files_2012.txt

Verifying...
  2018 timeseries: 100 files
  2012 timeseries: 100 files
  2018 weather   : 37 files
  2012 weather   : 37 files


## Cell 4: Metrics Helper Functions

In [44]:
def compute_metrics(y_true, y_pred, label=''):
    '''Compute all metrics. Uses positive values only for log metrics.'''
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    clean  = y_true > 0
    yt     = y_true[clean]
    yp     = y_pred[clean].clip(lower=0.001)
    mape_m = yt > 0.5
    return {
        'model' : label,
        'mae'   : mean_absolute_error(yt, yp),
        'rmse'  : np.sqrt(mean_squared_error(yt, yp)),
        'rmsle' : np.sqrt(mean_squared_log_error(yt, yp)),
        'r2'    : r2_score(yt, yp),
        'mape'  : np.mean(np.abs((yt[mape_m]-yp[mape_m])/yt[mape_m]))*100
                  if mape_m.sum() > 0 else np.nan,
        'n'     : int(clean.sum())
    }

def daily_metrics(df_in, pred_col, label=''):
    '''Aggregate to daily then compute metrics.'''
    d = df_in.copy()
    d['date'] = pd.to_datetime(d['timestamp']).dt.date
    daily = d.groupby(['building_id','date']).agg(
        total_kwh=(pred_col[0],'sum'), prediction=(pred_col[1],'sum')
    ).reset_index()
    return compute_metrics(daily['total_kwh'], daily['prediction'], label)

def print_metrics_table(metrics_list):
    '''Print a formatted comparison table.'''
    print(f'\n{"Model":<22} {"MAE":>8} {"RMSE":>8} {"RMSLE":>8} {"R²":>8} {"MAPE%":>8}')
    print('-'*68)
    for m in metrics_list:
        print(f'  {m["model"]:<20} {m["mae"]:>8.4f} {m["rmse"]:>8.4f} '
              f'{m["rmsle"]:>8.4f} {m["r2"]:>8.4f} {m["mape"]:>8.2f}')

print('✓ Metrics helpers defined')

✓ Metrics helpers defined


## Cell 5: Preprocessing Pipeline
Loads timeseries + weather, merges, outputs one combined parquet.
Automatically uses correct years based on `RUN_MODE`.

In [45]:
selected     = pd.read_csv(SELECTED)
weather_cache = {}

def load_weather(gisjoin, year):
    key = f'{gisjoin}_{year}'
    if key in weather_cache: return weather_cache[key]
    folder = WX_2018 if year == 2018 else WX_2012
    path   = folder / f'{gisjoin}_{year}.csv'
    if not path.exists():
        return None
    wx = pd.read_csv(path)
    ts_col = next((c for c in wx.columns if 'time' in c.lower() or 'date' in c.lower()), wx.columns[0])
    wx['timestamp'] = pd.to_datetime(wx[ts_col])
    if ts_col != 'timestamp': wx = wx.drop(columns=[ts_col])
    rename = {}
    for c in wx.columns:
        cl = c.lower()
        if 'dry bulb' in cl or 'dry_bulb' in cl: rename[c] = 'temp_c'
        elif 'relative humidity' in cl or 'humidity' in cl: rename[c] = 'humidity_pct'
        elif 'wind speed' in cl or 'wind_speed' in cl: rename[c] = 'wind_speed_ms'
        elif 'wind dir' in cl: rename[c] = 'wind_dir_deg'
        elif 'global horizontal' in cl or 'ghi' in cl: rename[c] = 'ghi_wm2'
        elif 'direct normal' in cl or 'dni' in cl: rename[c] = 'dni_wm2'
        elif 'diffuse' in cl or 'dhi' in cl: rename[c] = 'dhi_wm2'
    wx = wx.rename(columns=rename).set_index('timestamp')
    weather_cache[key] = wx
    return wx

def process_building(row, year):
    bid     = str(row['bid_str'])
    gisjoin = row['in.county']
    folder  = TS_2018 if year == 2018 else TS_2012
    ts_file = folder / f'{bid}-0.parquet'
    if not ts_file.exists(): return None, f'Not found: {ts_file.name}'
    try:
        df = pd.read_parquet(ts_file)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        app_cols = [c for c in df.columns
                    if 'out.electricity.' in c and 'energy_consumption' in c
                    and 'intensity' not in c]
        df['total_kwh'] = df[app_cols].sum(axis=1)
        df = (df.set_index('timestamp')[['total_kwh']]
                .resample('h').sum().reset_index())
        wx = load_weather(gisjoin, year)
        if wx is None: return None, f'No weather: {gisjoin} {year}'
        wx_h = wx.resample('h').mean().reset_index()
        wx_cols = ['timestamp'] + [c for c in wx_h.columns
                   if c in ['temp_c','humidity_pct','wind_speed_ms',
                            'wind_dir_deg','ghi_wm2','dni_wm2','dhi_wm2']]
        merged = df.merge(wx_h[wx_cols], on='timestamp', how='inner')
        merged['building_id']  = bid
        merged['gisjoin']      = gisjoin
        merged['climate_zone'] = row['climate_zone']
        merged['floor_area']   = row['in.geometry_floor_area']
        merged['bedrooms']     = row['in.bedrooms']
        merged['vintage']      = row['in.vintage']
        merged['hvac_cooling'] = row['in.hvac_cooling_type']
        merged['data_year']    = year
        if len(merged) < 365*24*0.90:
            return None, f'Insufficient rows: {len(merged)}'
        return merged, None
    except Exception as e:
        return None, str(e)[:80]

# Determine which years to process
years_to_process = [2018] if RUN_MODE == '1year' else [2012, 2018]
all_data, failed_log = [], []

for year in years_to_process:
    print(f'\nProcessing year {year}...')
    ok = 0
    for _, row in selected.iterrows():
        result, error = process_building(row, year)
        if result is not None:
            all_data.append(result)
            ok += 1
        else:
            failed_log.append({'year': year, 'bid': row['bid_str'], 'error': error})
    print(f'  ✓ {ok}/{len(selected)} buildings')

combined = pd.concat(all_data, ignore_index=True)
combined  = combined.sort_values(['building_id','timestamp']).reset_index(drop=True)

# Train/test split
tc = pd.Timestamp(TEST_CUTOFF)
if RUN_MODE == '1year':
    train = combined[combined['timestamp'] < tc].copy()
else:
    train = combined[(combined['data_year']==2012) |
                     ((combined['data_year']==2018) & (combined['timestamp']<tc))].copy()
test = combined[(combined['data_year']==2018) & (combined['timestamp']>=tc)].copy()

combined.to_parquet(PROCESSED/'preprocessed_data.parquet', index=False)
train.to_parquet(PROCESSED/'train.parquet', index=False)
test.to_parquet(PROCESSED/'test.parquet', index=False)

print(f'\n✓ Combined : {combined.shape}')
if RUN_MODE == "2year":
    year_counts = train["data_year"].value_counts().to_dict()
    print(f'  Train    : {train.shape}  {year_counts}')
else:
    print(f'  Train    : {train.shape}  ({len(train):,} rows)')
print(f'  Test     : {test.shape}')
print(f'  Mean kWh : {combined["total_kwh"].mean():.4f}')
print(f'  Saved to : {PROCESSED}')


Processing year 2012...
  ✓ 100/100 buildings

Processing year 2018...
  ✓ 100/100 buildings

✓ Combined : (1754400, 17)
  Train    : (1533500, 17)  {2012: 878400, 2018: 655100}
  Test     : (220900, 17)
  Mean kWh : 2.6133
  Saved to : run2_2year_r2/processed


In [46]:
print(f"RUN_MODE  : {RUN_MODE}")
print(f"TS_2018   : {TS_2018}  exists={TS_2018.exists()}")
print(f"TS_2012   : {TS_2012}  exists={TS_2012.exists()}")
print(f"WX_2018   : {WX_2018}  exists={WX_2018.exists()}")
print(f"WX_2012   : {WX_2012}  exists={WX_2012.exists()}")
print(f"SELECTED  : {SELECTED}  exists={SELECTED.exists()}")

ts_files = list(TS_2018.glob("*.parquet")) if TS_2018.exists() else []
wx_files = list(WX_2018.glob("*.csv")) if WX_2018.exists() else []
print(f"\nTimeseries files : {len(ts_files)}")
print(f"Weather files    : {len(wx_files)}")
if ts_files: print(f"Sample ts : {ts_files[0].name}")
if wx_files: print(f"Sample wx : {wx_files[0].name}")

selected = pd.read_csv(SELECTED)
first = selected.iloc[0]
bid = str(first['bid_str'])
gis = str(first['in.county'])
ts_path = TS_2018 / f"{bid}-0.parquet"
wx_path = WX_2018 / f"{gis}_2018.csv"
print(f"\nFirst building: {bid}")
print(f"  ts path    : {ts_path}")
print(f"  ts exists  : {ts_path.exists()}")
print(f"  wx path    : {wx_path}")
print(f"  wx exists  : {wx_path.exists()}")

RUN_MODE  : 2year
TS_2018   : data/timeseries_2018  exists=True
TS_2012   : data/timeseries_2012  exists=True
WX_2018   : data/weather_2018  exists=True
WX_2012   : data/weather_2012  exists=True
SELECTED  : selected_buildings.csv  exists=True

Timeseries files : 100
Weather files    : 37
Sample ts : 196172-0.parquet
Sample wx : G0600270_2018.csv

First building: 268737
  ts path    : data/timeseries_2018/268737-0.parquet
  ts exists  : True
  wx path    : data/weather_2018/G0600990_2018.csv
  wx exists  : True


In [47]:
# Run just the first building and print the error
first = selected.iloc[0]
result, error = process_building(first, 2018)
print(f"Result: {result is not None}")
print(f"Error : {error}")

# Also print failed_log to see all errors
print(f"\nFailed count: {len(failed_log)}")
if failed_log:
    print("First 5 errors:")
    for f in failed_log[:5]:
        print(f"  {f}")

Result: True
Error : None

Failed count: 0


## Cell 6: Feature Engineering

In [48]:
df = pd.read_parquet(PROCESSED/'preprocessed_data.parquet')
df = df.sort_values(['building_id','timestamp']).reset_index(drop=True)

# ════════════════════════════════════════════════════════════
# ENHANCED FEATURE ENGINEERING — Target R² > 0.93
# Key additions over original:
#   1. Hourly lags at 24h and 168h (same-hour yesterday / last week)
#   2. Deeper rolling windows at 24h, 48h, 168h
#   3. Weather × time interaction features
#   4. Building × weather interaction features
#   5. Demand volatility (rolling std at 24h, 168h)
#   6. Peak/off-peak flags
# ════════════════════════════════════════════════════════════

grp = ['building_id', 'data_year']

# ── 1. Cyclical time encoding (unchanged) ─────────────────────
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month']       = df['timestamp'].dt.month
df['week_of_year']= df['timestamp'].dt.isocalendar().week.astype(int)
df['hour_sin']    = np.sin(2*np.pi*df['hour']/24)
df['hour_cos']    = np.cos(2*np.pi*df['hour']/24)
df['day_sin']     = np.sin(2*np.pi*df['day_of_week']/7)
df['day_cos']     = np.cos(2*np.pi*df['day_of_week']/7)
df['month_sin']   = np.sin(2*np.pi*df['month']/12)
df['month_cos']   = np.cos(2*np.pi*df['month']/12)
df['week_sin']    = np.sin(2*np.pi*df['week_of_year']/52)
df['week_cos']    = np.cos(2*np.pi*df['week_of_year']/52)

# ── 2. Lag features — year-aware, no cross-year leakage ───────
# Original: lag 1,2,3,7,14 (hours)
# NEW: add lag 24 (same hour yesterday) and 168 (same hour last week)
# These are the strongest single predictors for hourly residential demand
for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
    df[f'kwh_lag_{lag}'] = df.groupby(grp)['total_kwh'].shift(lag)

for lag in [1, 2, 3, 24]:
    df[f'temp_lag_{lag}'] = df.groupby(grp)['temp_c'].shift(lag)

# ── 3. Rolling features — deeper windows ──────────────────────
# Original: 7h, 14h, 30h — too short for daily pattern capture
# NEW: 24h, 48h, 168h capture diurnal and weekly cycles explicitly
for w in [6, 24, 48, 168]:
    df[f'kwh_rolling_mean_{w}h'] = df.groupby(grp)['total_kwh'].transform(
        lambda x: x.rolling(w, min_periods=1).mean())
    df[f'kwh_rolling_std_{w}h']  = df.groupby(grp)['total_kwh'].transform(
        lambda x: x.rolling(w, min_periods=1).std().fillna(0))
    df[f'temp_rolling_mean_{w}h']= df.groupby(grp)['temp_c'].transform(
        lambda x: x.rolling(w, min_periods=1).mean())

# Demand delta: current vs 24h rolling mean (deviation from daily norm)
df['kwh_delta_24h']  = df['total_kwh'] - df['kwh_rolling_mean_24h']
df['kwh_delta_168h'] = df['total_kwh'] - df['kwh_rolling_mean_168h']

# ── 4. Calendar flags ─────────────────────────────────────────
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_monday']  = (df['day_of_week'] == 0).astype(int)
df['is_friday']  = (df['day_of_week'] == 4).astype(int)

holidays = pd.to_datetime([
    '2012-01-02','2012-05-28','2012-07-04','2012-09-03',
    '2012-11-22','2012-12-25',
    '2018-01-01','2018-05-28','2018-07-04','2018-09-03',
    '2018-11-22','2018-12-25'
])
df['is_holiday'] = df['timestamp'].dt.date.isin(holidays.date).astype(int)

# Peak / off-peak electricity pricing windows (CA TOU rates)
# Peak: weekday 16:00–21:00, Off-peak: 21:00–09:00, Mid-peak: rest
df['is_peak']     = ((df['hour'].between(16, 20)) & (df['is_weekend'] == 0)).astype(int)
df['is_off_peak'] = (df['hour'].isin([21,22,23,0,1,2,3,4,5,6,7,8])).astype(int)
df['is_morning_ramp'] = df['hour'].isin([5,6,7,8,9]).astype(int)    # fast ramp-up
df['is_evening_peak'] = df['hour'].isin([17,18,19,20]).astype(int)  # highest AC demand

df['season'] = df['month'].map({
    12:'winter',1:'winter',2:'winter',
    3:'spring',4:'spring',5:'spring',
    6:'summer',7:'summer',8:'summer',
    9:'fall',10:'fall',11:'fall'})

# ── 5. Derived weather features ───────────────────────────────
df['apparent_temp_c'] = (df['temp_c'] +
    0.33*(df['humidity_pct']/100*6.105*np.exp(17.27*df['temp_c']/(237.7+df['temp_c'])))
    - 0.70*df['wind_speed_ms'] - 4.00)
df['cdh']             = np.maximum(0, df['temp_c'] - 18.0)
df['hdh']             = np.maximum(0, 18.0 - df['temp_c'])
df['temp_squared']    = df['temp_c']**2
df['temp_x_humidity'] = df['temp_c'] * df['humidity_pct']
df['ghi_x_temp']      = df['ghi_wm2'] * df['temp_c']

# ── 6. NEW: Weather × time interaction features ───────────────
# These are the most impactful additions for R² improvement.
# They let the model know that 30°C at 14:00 drives far more AC
# than 30°C at 02:00 — a non-linear interaction GB can't easily
# discover without explicit encoding.
df['temp_x_hour_sin']     = df['temp_c'] * df['hour_sin']
df['temp_x_hour_cos']     = df['temp_c'] * df['hour_cos']
df['cdh_x_hour_sin']      = df['cdh']    * df['hour_sin']
df['cdh_x_hour_cos']      = df['cdh']    * df['hour_cos']
df['hdh_x_hour_sin']      = df['hdh']    * df['hour_sin']
df['hdh_x_hour_cos']      = df['hdh']    * df['hour_cos']
df['cdh_x_is_peak']       = df['cdh']    * df['is_peak']
df['cdh_x_is_weekend']    = df['cdh']    * df['is_weekend']
df['temp_x_lag24']        = df['temp_c'] * df['kwh_lag_24'].fillna(0)
df['ghi_x_hour_sin']      = df['ghi_wm2']* df['hour_sin']

# ── 7. NEW: Building × weather interaction features ───────────
# Converts floor_area from category to numeric for interactions
# Larger buildings have more thermal mass — larger CDH response
floor_area_map = {
    '0-499'   : 250,  '500-749' : 625,  '750-999' : 875,
    '1000-1499': 1250, '1500-1999': 1750, '2000-2499': 2250,
    '2500-2999': 2750, '3000+'   : 3500,
}
if 'floor_area' in df.columns:
    df['floor_area_num'] = df['floor_area'].map(floor_area_map).fillna(1250)
    df['floor_x_cdh']   = df['floor_area_num'] * df['cdh'] / 1000   # scale down
    df['floor_x_hdh']   = df['floor_area_num'] * df['hdh'] / 1000
    df['floor_x_lag24'] = df['floor_area_num'] * df['kwh_lag_24'].fillna(0) / 1000

# Vintage → numeric (older = worse insulation = more temperature sensitivity)
vintage_map = {
    '<1940':1935, '1940s':1945, '1950s':1955, '1960s':1965,
    '1970s':1975, '1980s':1985, '1990s':1995, '2000s':2005,
    '2010s':2015, '2015+':2017,
}
if 'vintage' in df.columns:
    df['vintage_num']   = df['vintage'].map(vintage_map).fillna(1985)
    df['vintage_age']   = 2018 - df['vintage_num']       # years old
    df['vintage_x_cdh'] = df['vintage_age'] * df['cdh'] / 100

# ── 8. One-hot encode categorical columns ─────────────────────
cat_cols = [c for c in ['season','floor_area','vintage','hvac_cooling']
            if c in df.columns]
df = pd.get_dummies(df, columns=cat_cols,
    prefix=[c.split('_')[0] for c in cat_cols], drop_first=False)

df.to_parquet(PROCESSED/'engineered_features.parquet', index=False)
feature_cols = [c for c in df.columns
                if c not in ['building_id','timestamp','total_kwh',
                             'gisjoin','climate_zone','data_year']]
print(f'✓ Enhanced features: {df.shape}')
print(f'  Feature count    : {len(feature_cols)}  (was ~73)')
print(f'  Key new features : kwh_lag_24, kwh_lag_168, kwh_lag_48,')
print(f'                     kwh_rolling_mean_168h, temp_x_hour_sin/cos,')
print(f'                     cdh_x_hour_sin/cos, floor_x_cdh, vintage_x_cdh')
print(f'  Missing values   : {df[feature_cols].isnull().sum().sum():,}')
print(f'  Saved to         : {PROCESSED}/engineered_features.parquet')


✓ Enhanced features: (1754400, 108)
  Feature count    : 102  (was ~73)
  Key new features : kwh_lag_24, kwh_lag_168, kwh_lag_48,
                     kwh_rolling_mean_168h, temp_x_hour_sin/cos,
                     cdh_x_hour_sin/cos, floor_x_cdh, vintage_x_cdh
  Missing values   : 58,800
  Saved to         : run2_2year_r2/processed/engineered_features.parquet


## Cell 7: Holt-Winters (Baseline)

In [49]:
df       = pd.read_parquet(PROCESSED/'preprocessed_data.parquet')
tc       = pd.Timestamp(TEST_CUTOFF)
if RUN_MODE == '1year':
    df_train = df[df['timestamp'] < tc].copy()
else:
    df_train = df[(df['data_year']==2012) |
                  ((df['data_year']==2018) & (df['timestamp']<tc))].copy()
df_test  = df[(df['data_year']==2018) & (df['timestamp']>=tc)].copy()

predictions_list, model_params = [], []

for idx, bid in enumerate(df['building_id'].unique(), 1):
    b_train = df_train[df_train['building_id']==bid]
    b_test  = df_test[df_test['building_id']==bid].copy()
    if len(b_train) < 100 or len(b_test) == 0: continue
    try:
        ts = b_train.set_index('timestamp')['total_kwh']
        m  = ExponentialSmoothing(ts, seasonal_periods=24,
                                   trend='add', seasonal='add',
                                   damped_trend=True).fit(optimized=True)
        b_test['hw_prediction'] = m.forecast(len(b_test)).values
        b_test['hw_residual']   = b_test['total_kwh'] - b_test['hw_prediction']
        predictions_list.append(b_test)
        model_params.append({'building_id':bid,
            'alpha':m.params['smoothing_level'],
            'beta':m.params['smoothing_trend'],
            'gamma':m.params['smoothing_seasonal']})
        if idx % 20 == 0: print(f'  [{idx}/100]')
    except Exception as e:
        print(f'  ✗ {bid}: {str(e)[:50]}')

hw_pred = pd.concat(predictions_list, ignore_index=True)
hw_pred.to_parquet(PRED_DIR/'holt_winters_predictions.parquet', index=False)
pd.DataFrame(model_params).to_csv(MODEL_DIR/'hw_params.csv', index=False)

m_hw = compute_metrics(hw_pred['total_kwh'], hw_pred['hw_prediction'], 'holt-Winters')
print(f'\nHolt-Winters: MAE={m_hw["mae"]:.4f}  RMSLE={m_hw["rmsle"]:.4f}  R²={m_hw["r2"]:.4f}')
print(f'✓ Saved: {PRED_DIR}/holt_winters_predictions.parquet')

  [20/100]
  [40/100]
  [60/100]
  [80/100]
  [100/100]

Holt-Winters: MAE=1.2727  RMSLE=0.5038  R²=0.0853
✓ Saved: run2_2year_r2/predictions/holt_winters_predictions.parquet


## Cell 8: Prophet Model

In [50]:
df       = pd.read_parquet(PROCESSED/'preprocessed_data.parquet')
tc       = pd.Timestamp(TEST_CUTOFF)
df['cdh'] = np.maximum(0, df['temp_c'] - 18.0)
df['hdh'] = np.maximum(0, 18.0 - df['temp_c'])

if RUN_MODE == '1year':
    df_train = df[df['timestamp'] < tc].copy()
else:
    df_train = df[(df['data_year']==2012) |
                  ((df['data_year']==2018) & (df['timestamp']<tc))].copy()
df_test = df[(df['data_year']==2018) & (df['timestamp']>=tc)].copy()

predictions_list, model_info = [], []

for idx, bid in enumerate(df['building_id'].unique(), 1):
    b_train = df_train[df_train['building_id']==bid].copy()
    b_test  = df_test[df_test['building_id']==bid].copy()
    if len(b_train) < 100 or len(b_test) == 0: continue
    try:
        b_train['log_kwh'] = np.log1p(b_train['total_kwh'].clip(lower=0))
        pt = b_train[['timestamp','log_kwh','temp_c','humidity_pct',
                       'ghi_wm2','cdh','hdh']].copy()
        pt.columns = ['ds','y','temp_c','humidity_pct','ghi_wm2','cdh','hdh']
        zone = b_train['climate_zone'].iloc[0]
        cdh_s = 2.0 if zone in ['2B','4B'] else 0.5 if zone=='5B' else 1.0
        hdh_s = 2.0 if zone=='5B' else 1.0
        m = Prophet(growth='linear', yearly_seasonality=False,
                    daily_seasonality=False, weekly_seasonality=True,
                    seasonality_prior_scale=0.1,
                    changepoint_prior_scale=0.005)
        m.add_seasonality('daily',   period=1,     fourier_order=10, prior_scale=0.1)
        m.add_seasonality('weekly',  period=7,     fourier_order=5,  prior_scale=0.05)
        m.add_seasonality('monthly', period=30.5,  fourier_order=3,  prior_scale=0.05)
        # Add yearly only for 2-year run — needs 2 cycles to fit reliably
        if RUN_MODE == '2year':
            m.add_seasonality('yearly', period=365.25, fourier_order=5, prior_scale=0.1)
        for reg, ps in [('temp_c',0.5),('humidity_pct',0.3),
                         ('ghi_wm2',0.5),('cdh',cdh_s),('hdh',hdh_s)]:
            m.add_regressor(reg, prior_scale=ps, standardize=True)
        m.fit(pt)
        future = b_test[['timestamp','temp_c','humidity_pct',
                           'ghi_wm2','cdh','hdh']].copy()
        future.columns = ['ds','temp_c','humidity_pct','ghi_wm2','cdh','hdh']
        fc   = m.predict(future)
        pred = np.expm1(fc['yhat'].values)
        tm   = b_train['total_kwh'].mean()
        pm   = pred.mean()
        if pm > 0.01: pred = pred * (tm/pm)
        pred = np.clip(pred, 0, b_train['total_kwh'].quantile(0.99)*1.5)
        b_test = b_test.copy()
        b_test['prophet_prediction'] = pred
        b_test['prophet_trend']      = np.expm1(fc['trend'].values)
        for comp in ['daily','weekly','monthly','yearly']:
            if comp in fc.columns:
                b_test[f'prophet_{comp}'] = fc[comp].values
        b_test['prophet_residual'] = b_test['total_kwh'] - b_test['prophet_prediction']
        predictions_list.append(b_test)
        mae_b = mean_absolute_error(b_test['total_kwh'], b_test['prophet_prediction'])
        model_info.append({'building_id':bid,'climate_zone':zone,'mae':mae_b})
        if idx % 10 == 0:
            print(f'  [{idx}/100] avg MAE: {np.mean([x["mae"] for x in model_info]):.3f}')
    except Exception as e:
        print(f'  ✗ {bid}: {str(e)[:60]}')

prophet_pred = pd.concat(predictions_list, ignore_index=True)
prophet_pred.to_parquet(PRED_DIR/'prophet_predictions.parquet', index=False)
pd.DataFrame(model_info).to_csv(MODEL_DIR/'prophet_info.csv', index=False)
m_p = compute_metrics(prophet_pred['total_kwh'], prophet_pred['prophet_prediction'], 'Prophet')
print(f'\nProphet: MAE={m_p["mae"]:.4f}  RMSLE={m_p["rmsle"]:.4f}  R²={m_p["r2"]:.4f}')
print(f'✓ Saved: {PRED_DIR}/prophet_predictions.parquet')

18:21:24 - cmdstanpy - INFO - Chain [1] start processing
18:21:25 - cmdstanpy - INFO - Chain [1] done processing
18:21:26 - cmdstanpy - INFO - Chain [1] start processing
18:21:26 - cmdstanpy - INFO - Chain [1] done processing
18:21:27 - cmdstanpy - INFO - Chain [1] start processing
18:21:28 - cmdstanpy - INFO - Chain [1] done processing
18:21:29 - cmdstanpy - INFO - Chain [1] start processing
18:21:30 - cmdstanpy - INFO - Chain [1] done processing
18:21:31 - cmdstanpy - INFO - Chain [1] start processing
18:21:31 - cmdstanpy - INFO - Chain [1] done processing
18:21:32 - cmdstanpy - INFO - Chain [1] start processing
18:21:33 - cmdstanpy - INFO - Chain [1] done processing
18:21:34 - cmdstanpy - INFO - Chain [1] start processing
18:21:35 - cmdstanpy - INFO - Chain [1] done processing
18:21:36 - cmdstanpy - INFO - Chain [1] start processing
18:21:36 - cmdstanpy - INFO - Chain [1] done processing
18:21:37 - cmdstanpy - INFO - Chain [1] start processing
18:21:38 - cmdstanpy - INFO - Chain [1]

  [10/100] avg MAE: 0.854


18:21:40 - cmdstanpy - INFO - Chain [1] start processing
18:21:41 - cmdstanpy - INFO - Chain [1] done processing
18:21:42 - cmdstanpy - INFO - Chain [1] start processing
18:21:42 - cmdstanpy - INFO - Chain [1] done processing
18:21:43 - cmdstanpy - INFO - Chain [1] start processing
18:21:44 - cmdstanpy - INFO - Chain [1] done processing
18:21:45 - cmdstanpy - INFO - Chain [1] start processing
18:21:46 - cmdstanpy - INFO - Chain [1] done processing
18:21:47 - cmdstanpy - INFO - Chain [1] start processing
18:21:47 - cmdstanpy - INFO - Chain [1] done processing
18:21:48 - cmdstanpy - INFO - Chain [1] start processing
18:21:49 - cmdstanpy - INFO - Chain [1] done processing
18:21:50 - cmdstanpy - INFO - Chain [1] start processing
18:21:50 - cmdstanpy - INFO - Chain [1] done processing
18:21:51 - cmdstanpy - INFO - Chain [1] start processing
18:21:52 - cmdstanpy - INFO - Chain [1] done processing
18:21:53 - cmdstanpy - INFO - Chain [1] start processing
18:21:53 - cmdstanpy - INFO - Chain [1]

  [20/100] avg MAE: 0.726


18:21:56 - cmdstanpy - INFO - Chain [1] start processing
18:21:56 - cmdstanpy - INFO - Chain [1] done processing
18:21:57 - cmdstanpy - INFO - Chain [1] start processing
18:21:58 - cmdstanpy - INFO - Chain [1] done processing
18:21:59 - cmdstanpy - INFO - Chain [1] start processing
18:22:00 - cmdstanpy - INFO - Chain [1] done processing
18:22:01 - cmdstanpy - INFO - Chain [1] start processing
18:22:02 - cmdstanpy - INFO - Chain [1] done processing
18:22:03 - cmdstanpy - INFO - Chain [1] start processing
18:22:03 - cmdstanpy - INFO - Chain [1] done processing
18:22:04 - cmdstanpy - INFO - Chain [1] start processing
18:22:05 - cmdstanpy - INFO - Chain [1] done processing
18:22:06 - cmdstanpy - INFO - Chain [1] start processing
18:22:07 - cmdstanpy - INFO - Chain [1] done processing
18:22:07 - cmdstanpy - INFO - Chain [1] start processing
18:22:08 - cmdstanpy - INFO - Chain [1] done processing
18:22:09 - cmdstanpy - INFO - Chain [1] start processing
18:22:10 - cmdstanpy - INFO - Chain [1]

...

  [80/100] avg MAE: 0.828


18:23:34 - cmdstanpy - INFO - Chain [1] start processing
18:23:34 - cmdstanpy - INFO - Chain [1] done processing
18:23:35 - cmdstanpy - INFO - Chain [1] start processing
18:23:36 - cmdstanpy - INFO - Chain [1] done processing
18:23:37 - cmdstanpy - INFO - Chain [1] start processing
18:23:37 - cmdstanpy - INFO - Chain [1] done processing
18:23:38 - cmdstanpy - INFO - Chain [1] start processing
18:23:39 - cmdstanpy - INFO - Chain [1] done processing
18:23:40 - cmdstanpy - INFO - Chain [1] start processing
18:23:41 - cmdstanpy - INFO - Chain [1] done processing
18:23:41 - cmdstanpy - INFO - Chain [1] start processing
18:23:42 - cmdstanpy - INFO - Chain [1] done processing
18:23:43 - cmdstanpy - INFO - Chain [1] start processing
18:23:44 - cmdstanpy - INFO - Chain [1] done processing
18:23:45 - cmdstanpy - INFO - Chain [1] start processing
18:23:45 - cmdstanpy - INFO - Chain [1] done processing
18:23:46 - cmdstanpy - INFO - Chain [1] start processing
18:23:47 - cmdstanpy - INFO - Chain [1]

  [90/100] avg MAE: 0.883


18:23:50 - cmdstanpy - INFO - Chain [1] start processing
18:23:50 - cmdstanpy - INFO - Chain [1] done processing
18:23:51 - cmdstanpy - INFO - Chain [1] start processing
18:23:52 - cmdstanpy - INFO - Chain [1] done processing
18:23:53 - cmdstanpy - INFO - Chain [1] start processing
18:23:54 - cmdstanpy - INFO - Chain [1] done processing
18:23:55 - cmdstanpy - INFO - Chain [1] start processing
18:23:55 - cmdstanpy - INFO - Chain [1] done processing
18:23:56 - cmdstanpy - INFO - Chain [1] start processing
18:23:57 - cmdstanpy - INFO - Chain [1] done processing
18:23:58 - cmdstanpy - INFO - Chain [1] start processing
18:23:59 - cmdstanpy - INFO - Chain [1] done processing
18:24:00 - cmdstanpy - INFO - Chain [1] start processing
18:24:01 - cmdstanpy - INFO - Chain [1] done processing
18:24:02 - cmdstanpy - INFO - Chain [1] start processing
18:24:02 - cmdstanpy - INFO - Chain [1] done processing
18:24:03 - cmdstanpy - INFO - Chain [1] start processing
18:24:04 - cmdstanpy - INFO - Chain [1]

  [100/100] avg MAE: 0.893

Prophet: MAE=0.8792  RMSLE=0.3240  R²=0.5709
✓ Saved: run2_2year_r2/predictions/prophet_predictions.parquet


## Cell 9: Gradient Boost

In [ ]:
df  = pd.read_parquet(PROCESSED/'engineered_features.parquet')
tc  = pd.Timestamp(TEST_CUTOFF)
exc = ['building_id','timestamp','total_kwh','gisjoin','climate_zone','data_year',
       'floor_area_num','vintage_num','vintage_age',  # numeric intermediates — exclude
       'hour','day_of_week','month','week_of_year',   # raw cyclical — encoded versions used
       'kwh_delta_24h','kwh_delta_168h',              # computed from target — exclude to avoid leakage
       ]
feature_cols = [c for c in df.columns if c not in exc]

if RUN_MODE == '1year':
    df_train = df[df['timestamp'] < tc].dropna(subset=feature_cols)
else:
    df_train = df[(df['data_year']==2012) |
                  ((df['data_year']==2018) & (df['timestamp']<tc))].dropna(subset=feature_cols)
df_test = df[(df['data_year']==2018) & (df['timestamp']>=tc)].dropna(subset=feature_cols).copy()

X_train = df_train[feature_cols]
y_train = df_train['total_kwh']
X_test  = df_test[feature_cols]
y_test  = df_test['total_kwh']
print(f'Train: {X_train.shape}  Test: {X_test.shape}  Features: {len(feature_cols)}')

# ── Fix libomp.dylib path for LightGBM ──
import os
# FIXED: removed hardcoded macOS-only path (was breaking on any
# machine other than the original author's laptop). LightGBM does
# not require this on Linux or Windows; on Mac it is only needed if
# libomp was installed via Homebrew, in which case 'brew install libomp'
# typically registers it correctly without manual path injection.

# ── Try LightGBM first (5-8x faster, usually +0.01-0.02 R²) ──
try:
    import lightgbm as lgb
    USE_LGBM = True
    print('✓ LightGBM available — using LightGBM')
except ImportError:
    USE_LGBM = False
    print('⚠ LightGBM not found — falling back to sklearn GB')
    print('  Install with: pip install lightgbm --break-system-packages')

if USE_LGBM:
    model = lgb.LGBMRegressor(
        n_estimators       = 500,
        learning_rate      = 0.05,
        max_depth          = 8,
        num_leaves         = 63,       # 2^(max_depth-1) — standard rule
        min_child_samples  = 20,
        subsample          = 0.8,
        colsample_bytree   = 0.7,
        reg_alpha          = 0.1,      # L1 regularisation
        reg_lambda         = 1.0,      # L2 regularisation
        random_state       = 42,
        n_jobs             = -1,
        verbose            = -1,
    )
    model.fit(X_train, y_train)
    df_test['gb_prediction'] = model.predict(X_test)
    print(f'✓ LightGBM trained')
else:
    model = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        min_samples_split=10,
        min_samples_leaf=5,
        subsample=0.8,
        random_state=42
    )
    model.fit(X_train, y_train)
    df_test['gb_prediction'] = model.predict(X_test)
    print(f'✓ sklearn GradientBoosting trained')

m_gb = compute_metrics(y_test, df_test['gb_prediction'], 'Gradient Boost')
print(f'GB: MAE={m_gb["mae"]:.4f}  RMSLE={m_gb["rmsle"]:.4f}  R²={m_gb["r2"]:.4f}')

df_test.to_parquet(PRED_DIR/'gb_predictions.parquet', index=False)
print(f'✓ Saved: {PRED_DIR}/gb_predictions.parquet')

# Save trained model
with open(MODEL_DIR/'gb_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print(f'✓ Saved: {MODEL_DIR}/gb_model.pkl')

Train: (1499900, 93)  Test: (220900, 93)  Features: 93
✓ LightGBM available — using LightGBM
✓ LightGBM trained
GB: MAE=0.3487  RMSLE=0.1473  R²=0.9130
✓ Saved: run2_2year_r2/predictions/gb_predictions.parquet
✓ Saved: run2_2year_r2/models/gb_model.pkl


In [57]:
import pandas as pd
from pathlib import Path

# Check which parquet file GB actually loaded
for run in ['run1_1year', 'run2_2year', 'run1_1year_r2', 'run2_2year_r2']:
    p = Path(run) / 'processed' / 'engineered_features.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        print(f'{run}: {df.shape[1]} columns')

run1_1year: 73 columns
run2_2year: 73 columns
run1_1year_r2: 108 columns
run2_2year_r2: 108 columns


## Cell 10: Ensemble Forecast

In [52]:
prophet_df = pd.read_parquet(PRED_DIR/'prophet_predictions.parquet')
gb_df      = pd.read_parquet(PRED_DIR/'gb_predictions.parquet')
hw_df      = pd.read_parquet(PRED_DIR/'holt_winters_predictions.parquet')

# Align — use gb total_kwh as ground truth (consistent release)
prophet_df['building_id'] = prophet_df['building_id'].astype(str)
gb_df['building_id']      = gb_df['building_id'].astype(str)

df = gb_df[['building_id','timestamp','total_kwh','climate_zone','gb_prediction']].merge(
    prophet_df[['building_id','timestamp','prophet_prediction']],
    on=['building_id','timestamp'], how='inner')
print(f'Merged: {df.shape}')

# Grid search optimal weights per zone
clean_df = df[df['total_kwh'] > 0].copy()
best_weights = {}

print('\nOptimising weights per zone...')
for zone in sorted(df['climate_zone'].dropna().unique()):
    z = clean_df[clean_df['climate_zone']==zone]
    best_r, best_w = 999, 0.0
    for w_p in np.arange(0.0, 0.51, 0.025):
        pred = (w_p*z['prophet_prediction']+(1-w_p)*z['gb_prediction']).clip(lower=0.001)
        r = np.sqrt(mean_squared_log_error(z['total_kwh'], pred))
        if r < best_r: best_r, best_w = r, w_p
    best_weights[zone] = {'prophet': best_w, 'gb': round(1-best_w,3)}
    print(f'  Zone {zone}: P={best_w:.3f} GB={1-best_w:.3f}  RMSLE={best_r:.4f}')

df['w_p']  = df['climate_zone'].apply(lambda z: best_weights.get(z,{'prophet':0})['prophet'])
df['w_gb'] = df['climate_zone'].apply(lambda z: best_weights.get(z,{'gb':1})['gb'])
df['ensemble_prediction'] = (
    df['w_p']*df['prophet_prediction'] + df['w_gb']*df['gb_prediction']
).clip(lower=0)

# All metrics
m_hw  = compute_metrics(hw_df['total_kwh'],    hw_df['hw_prediction'],       'holt-Winters')
m_p   = compute_metrics(df['total_kwh'],       df['prophet_prediction'],      'Prophet')
m_gb  = compute_metrics(df['total_kwh'],       df['gb_prediction'],           'Gradient Boost')
m_ens = compute_metrics(df['total_kwh'],       df['ensemble_prediction'],     'Ensemble')

print('\n' + '='*68)
print(f'FINAL COMPARISON ({RUN_MODE.upper()})')
print('='*68)
print_metrics_table([m_hw, m_p, m_gb, m_ens])

best_stat = min(m_hw['rmsle'], m_p['rmsle'])
imp = (best_stat - m_ens['rmsle']) / best_stat * 100
print(f'\nRMSLE improvement vs best statistical baseline: {imp:.1f}%')
print(f'Proposal target ≥15%: {"✓ ACHIEVED" if imp>=15 else "⚠ not met"}')

df.to_parquet(PRED_DIR/'ensemble_predictions.parquet', index=False)
pd.DataFrame([m_hw,m_p,m_gb,m_ens]).to_csv(RESULTS/'metrics_summary.csv', index=False)
pd.DataFrame([{'zone':z,**w} for z,w in best_weights.items()])\
    .to_csv(RESULTS/'ensemble_weights.csv', index=False)
print(f'\n✓ Saved to {RESULTS}')

Merged: (220900, 6)

Optimising weights per zone...
  Zone 2B: P=0.025 GB=0.975  RMSLE=0.1230
  Zone 3B: P=0.000 GB=1.000  RMSLE=0.1360
  Zone 3C: P=0.000 GB=1.000  RMSLE=0.1586
  Zone 4B: P=0.000 GB=1.000  RMSLE=0.1615
  Zone 4C: P=0.025 GB=0.975  RMSLE=0.1420
  Zone 5B: P=0.000 GB=1.000  RMSLE=0.1499

FINAL COMPARISON (2YEAR)

Model                       MAE     RMSE    RMSLE       R²    MAPE%
--------------------------------------------------------------------
  holt-Winters           1.2727   2.5892   0.5038   0.0853    51.54
  Prophet                0.8792   1.7733   0.3240   0.5709    38.91
  Gradient Boost         0.3487   0.7983   0.1473   0.9130    15.19
  Ensemble               0.3491   0.7984   0.1473   0.9130    15.20

RMSLE improvement vs best statistical baseline: 54.5%
Proposal target ≥15%: ✓ ACHIEVED

✓ Saved to run2_2year_r2/results


## Cell 11: XAI Analysis (SHAP + LIME + Anchors)

In [65]:
with open(MODEL_DIR/'gb_model.pkl','rb') as f: gb_model = pickle.load(f)
feature_cols = gb_model.feature_names_in_  
print(f'Feature count from model: {len(feature_cols)}')
print(MODEL_DIR)


Feature count from model: 93
run2_2year_r2/models


In [ ]:
)

Adding 4 missing columns as zeros: [np.str_('hvac_Central_AC'), np.str_('hvac_Ducted_Heat_Pump'), np.str_('hvac_Non-Ducted_Heat_Pump'), np.str_('hvac_Room_AC')]
✓ All 93 model features present in dataframe


In [72]:
import shap
from lime.lime_tabular import LimeTabularExplainer

# ── Load model and data ───────────────────────────────────────
with open(MODEL_DIR/'gb_model.pkl','rb') as f: gb_model = pickle.load(f)
df = pd.read_parquet(PROCESSED/'engineered_features.parquet')
df = df.sort_values(['building_id','timestamp']).reset_index(drop=True)

# Re-attach climate_zone from selected_buildings
zone_map = pd.read_csv(SELECTED).set_index('bid_str')['climate_zone'].to_dict()
df['building_id']  = df['building_id'].astype(str)
df['climate_zone'] = df['building_id'].map({str(k):v for k,v in zone_map.items()})
print(f"climate_zone: {df['climate_zone'].value_counts().to_dict()}")
# Ensure all model feature columns exist in df
# Add missing one-hot columns as zeros


feature_cols = list(gb_model.feature_names_in_) # XGBoost / sklearn
print(f'Feature count from model: {len(feature_cols)}')  # should print 93 or 102
missing_cols = [c for c in feature_cols if c not in df.columns]
if missing_cols:
    print(f'Adding {len(missing_cols)} missing columns as zeros: {missing_cols}')
    for c in missing_cols:
        df[c] = 0

# Confirm all features now present
assert all(c in df.columns for c in feature_cols), "Still missing features"
print(f'✓ All {len(feature_cols)} model features present in dataframe')

# Feature columns
# Always derive feature_cols from the saved model — never from the dataframe
# Test set
tc      = pd.Timestamp(TEST_CUTOFF)
df_test = df[(df['data_year']==2018) & (df['timestamp']>=tc)].dropna(subset=feature_cols).reset_index(drop=True)
X_test  = df_test[feature_cols]

# Verify climate_zone survived the filter
print(f"climate_zone in df_test: {'climate_zone' in df_test.columns}")
print(f"df_test zones: {df_test['climate_zone'].value_counts().to_dict()}")

# Stratified XAI sample — 200 per zone
# Stratified sample — avoid groupby.apply which drops climate_zone in pandas 3.x
xai_frames = []
for zone in df_test['climate_zone'].dropna().unique():
    zone_df = df_test[df_test['climate_zone'] == zone]
    n       = min(len(zone_df), 200)
    xai_frames.append(zone_df.sample(n=n, random_state=42))

xai_df = pd.concat(xai_frames).reset_index(drop=True)
X_samp = xai_df[feature_cols].reset_index(drop=True)
y_samp = xai_df['total_kwh'].reset_index(drop=True)

print(f"climate_zone in xai_df: {'climate_zone' in xai_df.columns}")
print(f"xai_df zones: {xai_df['climate_zone'].value_counts().to_dict()}")
print(f"XAI sample: {xai_df.shape}")

# Verify climate_zone survived
print(f"climate_zone in xai_df: {'climate_zone' in xai_df.columns}")
print(f"xai_df zones: {xai_df['climate_zone'].value_counts().to_dict()}")
X_samp = xai_df[feature_cols].reset_index(drop=True)
y_samp = xai_df['total_kwh'].reset_index(drop=True)
print(f'XAI sample: {xai_df.shape}  zones: {xai_df["climate_zone"].value_counts().to_dict()}')

# ── SHAP ──────────────────────────────────────────────────────
print('\nComputing SHAP...')
explainer   = shap.TreeExplainer(gb_model)
shap_values = explainer.shap_values(X_samp)
shap_imp = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_imp.to_csv(RESULTS/'xai'/'shap'/'global_importance.csv', index=False)

print('Top 5 SHAP features:')
for _, r in shap_imp.head(5).iterrows():
    print(f'  {r["feature"]:<35} {r["mean_abs_shap"]:.4f}')

# Per-zone SHAP heatmap
zone_shap = {}
for zone, grp in xai_df.groupby('climate_zone'):
    mask = xai_df['climate_zone'].values == zone
    zone_shap[zone] = np.abs(shap_values[mask]).mean(axis=0)
pd.DataFrame(zone_shap, index=feature_cols).T\
    .to_csv(RESULTS/'xai'/'shap'/'zone_shap_importance.csv')

# SHAP global plot
fig, ax = plt.subplots(figsize=(10,7))
top15 = shap_imp.head(15)
ax.barh(range(len(top15)), top15['mean_abs_shap'].values[::-1],
        color='#1D9E75', edgecolor='white')
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['feature'].values[::-1], fontsize=10)
ax.set_xlabel('Mean |SHAP| (kWh)')
ax.set_title(f'Global Feature Importance — SHAP ({RUN_MODE})', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS/'xai'/'plots'/'shap_global.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ SHAP complete')

# ── LIME ──────────────────────────────────────────────────────
print('\nComputing LIME...')
lime_exp = LimeTabularExplainer(
    X_test.values, feature_names=feature_cols,
    mode='regression', random_state=42, discretize_continuous=True)
lime_results = []
for zone, grp in xai_df.groupby('climate_zone'):
    grp = grp.reset_index(drop=True)
    for label, row_i in [('low',  int(grp['total_kwh'].idxmin())),
                          ('mid',  len(grp)//2),
                          ('high', int(grp['total_kwh'].idxmax()))]:
        inst = X_samp.iloc[row_i].values
        try:
            exp = lime_exp.explain_instance(
                inst, gb_model.predict, num_features=8, num_samples=500)
            rec = {'zone': zone, 'level': label,
                   'actual_kwh':    float(grp.loc[row_i,'total_kwh']),
                   'predicted_kwh': float(gb_model.predict(inst.reshape(1,-1))[0])}
            for j, (rule, impact) in enumerate(exp.as_list()[:5], 1):
                rec[f'rule_{j}']   = rule
                rec[f'impact_{j}'] = round(float(impact), 4)
            lime_results.append(rec)
        except Exception as e:
            print(f'    {zone} {label} failed: {str(e)[:50]}')
    print(f'  Zone {zone}: ✓')
pd.DataFrame(lime_results).to_csv(
    RESULTS/'xai'/'lime'/'lime_explanations.csv', index=False)
print('✓ LIME complete')

# ── Anchors ───────────────────────────────────────────────────
print('\nComputing Anchors...')
y_pred_all    = gb_model.predict(X_test)
low_t, high_t = np.percentile(y_pred_all, 33), np.percentile(y_pred_all, 67)

def predict_class(X):
    p = gb_model.predict(X)
    return np.where(p <= low_t, 0, np.where(p <= high_t, 1, 2))

try:
    from anchor import anchor_tabular
    anch = anchor_tabular.AnchorTabularExplainer(
        ['low','mid','high'], feature_cols,
        X_test.values, discretizer='quartile')
    anchor_results = []
    for zone, grp in xai_df.groupby('climate_zone'):
        hi   = int(grp['total_kwh'].idxmax())
        inst = X_samp.iloc[hi].values
        try:
            exp  = anch.explain_instance(
                inst, predict_class,
                threshold=0.95, delta=0.1, beam_size=2, max_anchor_size=4)
            rule = ' AND '.join(exp.names())
            print(f'  Zone {zone}: IF {rule} (precision={exp.precision():.0%})')
            anchor_results.append({
                'zone': zone, 'rule': rule,
                'precision': round(exp.precision(), 3),
                'coverage':  round(exp.coverage(), 3)
            })
        except Exception as e:
            print(f'  Zone {zone}: failed — {str(e)[:60]}')
    pd.DataFrame(anchor_results).to_csv(
        RESULTS/'xai'/'anchors'/'anchor_rules.csv', index=False)
    print('✓ Anchors complete')
except ImportError:
    print('  anchor-exp not installed — pip install anchor-exp')

print(f'\n✓ All XAI outputs saved to {RESULTS}/xai/')

climate_zone: {'3B': 614040, '3C': 438600, '4B': 263160, '4C': 175440, '5B': 175440, '2B': 87720}
Feature count from model: 93
Adding 4 missing columns as zeros: [np.str_('hvac_Central_AC'), np.str_('hvac_Ducted_Heat_Pump'), np.str_('hvac_Non-Ducted_Heat_Pump'), np.str_('hvac_Room_AC')]
✓ All 93 model features present in dataframe
climate_zone in df_test: True
df_test zones: {'3B': 77315, '3C': 55225, '4B': 33135, '4C': 22090, '5B': 22090, '2B': 11045}
climate_zone in xai_df: True
xai_df zones: {'4C': 200, '3B': 200, '3C': 200, '4B': 200, '5B': 200, '2B': 200}
XAI sample: (1200, 112)
climate_zone in xai_df: True
xai_df zones: {'4C': 200, '3B': 200, '3C': 200, '4B': 200, '5B': 200, '2B': 200}
XAI sample: (1200, 112)  zones: {'4C': 200, '3B': 200, '3C': 200, '4B': 200, '5B': 200, '2B': 200}

Computing SHAP...
Top 5 SHAP features:
  kwh_rolling_mean_6h                 0.7070
  kwh_lag_1                           0.6609
  kwh_rolling_std_6h                  0.2487
  kwh_lag_3              

## Cell 12: Evaluation Metrics — Final Comparison

In [55]:
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.gridspec as gridspec

# Load all predictions
ensemble_df = pd.read_parquet(PRED_DIR/'ensemble_predictions.parquet')
hw_df       = pd.read_parquet(PRED_DIR/'holt_winters_predictions.parquet')
prophet_df  = pd.read_parquet(PRED_DIR/'prophet_predictions.parquet')

# ── Baseline 1: Persistence ───────────────────────────────────
preprocessed = pd.read_parquet(PROCESSED/'preprocessed_data.parquet')
test_data = preprocessed[(preprocessed['data_year']==2018) &
                           (preprocessed['timestamp']>=pd.Timestamp(TEST_CUTOFF))].copy()
test_data = test_data.sort_values(['building_id','timestamp'])
test_data['persistence'] = test_data.groupby('building_id')['total_kwh'].shift(1)
test_data['seasonal_naive'] = test_data.groupby('building_id')['total_kwh'].shift(24)
persist_df = test_data.dropna(subset=['persistence'])
sn_df      = test_data.dropna(subset=['seasonal_naive'])

m_persist = compute_metrics(persist_df['total_kwh'], persist_df['persistence'],     'Persistence')
m_sn      = compute_metrics(sn_df['total_kwh'],      sn_df['seasonal_naive'],       'Seasonal Naive')
m_hw      = compute_metrics(hw_df['total_kwh'],       hw_df['hw_prediction'],        'holt-Winters')
m_p       = compute_metrics(prophet_df['total_kwh'],  prophet_df['prophet_prediction'],'Prophet')
m_gb      = compute_metrics(ensemble_df['total_kwh'], ensemble_df['gb_prediction'],  'Gradient Boost')
m_ens     = compute_metrics(ensemble_df['total_kwh'], ensemble_df['ensemble_prediction'],'Ensemble')

all_metrics = [m_persist, m_sn, m_hw, m_p, m_gb, m_ens]

print(f'{'='*68}')
print(f'FINAL EVALUATION — {RUN_MODE.upper()}')
print(f'{'='*68}')
print_metrics_table(all_metrics)

# RMSLE improvements
print(f'\nRMSLE improvements of Ensemble over baselines:')
for m in [m_persist, m_sn, m_hw, m_p]:
    imp = (m['rmsle'] - m_ens['rmsle']) / m['rmsle'] * 100
    print(f'  vs {m["model"]:<18}: {imp:>6.1f}%  {"✓" if imp>=15 else "✗"}')

# Daily metrics
print(f'\nDAILY AGGREGATE:')
for pred_col, label in [('hw_prediction','hW'),
                          ('prophet_prediction','Prophet')]:
    try:
        df_tmp = pd.read_parquet(PRED_DIR/f'{"holt_winters" if "hw" in pred_col else "prophet"}_predictions.parquet')
        dm = daily_metrics(df_tmp, ('total_kwh', pred_col), label)
        print(f'  {label:<15} RMSLE={dm["rmsle"]:.4f}  R²={dm["r2"]:.4f}')
    except: pass
for pred_col, label in [('gb_prediction','GB'),('ensemble_prediction','Ensemble')]:
    dm = daily_metrics(ensemble_df, ('total_kwh', pred_col), label)
    print(f'  {label:<15} RMSLE={dm["rmsle"]:.4f}  R²={dm["r2"]:.4f}')

# ── Plots ─────────────────────────────────────────────────────
results_df = pd.DataFrame(all_metrics)

fig, axes = plt.subplots(1, 2, figsize=(14,5))
valid = results_df.dropna(subset=['rmsle'])
colors = ['#1D9E75' if r=='Ensemble' else '#9FE1CB' if r=='Gradient Boost'
          else '#E0E0E0' for r in valid['model']]
axes[0].bar(valid['model'], valid['rmsle'], color=colors, edgecolor='white')
axes[0].set_title(f'RMSLE Comparison ({RUN_MODE})', fontweight='bold')
axes[0].set_xticklabels(valid['model'], rotation=20, ha='right')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)
for bar, val in zip(axes[0].patches, valid['rmsle']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.3f}', ha='center', fontsize=8)

axes[1].bar(valid['model'], valid['r2'], color=colors, edgecolor='white')
axes[1].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[1].set_title(f'R² Comparison ({RUN_MODE})', fontweight='bold')
axes[1].set_xticklabels(valid['model'], rotation=20, ha='right')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS/'evaluation'/'plots'/f'model_comparison_{RUN_MODE}.png',
            dpi=150, bbox_inches='tight')
plt.close()

# Actual vs predicted
sample = ensemble_df.sample(500, random_state=42)
fig, ax = plt.subplots(figsize=(7,7))
ax.scatter(sample['total_kwh'], sample['ensemble_prediction'],
           alpha=0.4, s=15, color='#1D9E75')
mx = max(sample['total_kwh'].max(), sample['ensemble_prediction'].max())
ax.plot([0,mx],[0,mx],'r--',linewidth=1)
ax.set_xlabel('Actual (kWh)'); ax.set_ylabel('Predicted (kWh)')
ax.set_title(f'Actual vs Predicted — Ensemble ({RUN_MODE})\nRMSLE={m_ens["rmsle"]:.4f}  R²={m_ens["r2"]:.4f}',
             fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS/'evaluation'/'plots'/f'actual_vs_predicted_{RUN_MODE}.png',
            dpi=150, bbox_inches='tight')
plt.close()

# Save
results_df.to_csv(RESULTS/'evaluation'/f'all_metrics_{RUN_MODE}.csv', index=False)
print(f'\n✓ Plots saved  : {RESULTS}/evaluation/plots/')
print(f'✓ Metrics saved: {RESULTS}/evaluation/all_metrics_{RUN_MODE}.csv')

FINAL EVALUATION — 2YEAR

Model                       MAE     RMSE    RMSLE       R²    MAPE%
--------------------------------------------------------------------
  Persistence            0.5989   1.4527   0.2434   0.7121    23.23
  Seasonal Naive         0.7456   1.8097   0.3027   0.5528    30.03
  holt-Winters           1.2727   2.5892   0.5038   0.0853    51.54
  Prophet                0.8792   1.7733   0.3240   0.5709    38.91
  Gradient Boost         0.3487   0.7983   0.1473   0.9130    15.19
  Ensemble               0.3491   0.7984   0.1473   0.9130    15.20

RMSLE improvements of Ensemble over baselines:
  vs Persistence       :   39.5%  ✓
  vs Seasonal Naive    :   51.3%  ✓
  vs holt-Winters      :   70.8%  ✓
  vs Prophet           :   54.5%  ✓

DAILY AGGREGATE:
  hW              RMSLE=0.7010  R²=0.1303
  Prophet         RMSLE=0.4480  R²=0.8083
  GB              RMSLE=0.0988  R²=0.9909
  Ensemble        RMSLE=0.0993  R²=0.9908

✓ Plots saved  : run2_2year_r2/results/evaluation/

##$ HYBRID 

In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer

# ── Load model and data ───────────────────────────────────────
with open(MODEL_DIR/'gb_model.pkl','rb') as f: gb_model = pickle.load(f)
df = pd.read_parquet(PROCESSED/'engineered_features.parquet')
df = df.sort_values(['building_id','timestamp']).reset_index(drop=True)

# Feature columns — MUST match training cell exactly
exc = ['building_id','timestamp','total_kwh','gisjoin','climate_zone','data_year',
       'floor_area_num','vintage_num','vintage_age',  # numeric intermediates — exclude
       'hour','day_of_week','month','week_of_year',   # raw cyclical — encoded versions used
       'kwh_delta_24h','kwh_delta_168h',              # computed from target — exclude to avoid leakage
       ]
feature_cols = [c for c in df.columns if c not in exc]

# Test set
tc      = pd.Timestamp(TEST_CUTOFF)
df_test = df[(df['data_year']==2018) & (df['timestamp']>=tc)].dropna(subset=feature_cols).reset_index(drop=True)
X_test  = df_test[feature_cols]

# Verify climate_zone survived the filter
print(f"climate_zone in df_test: {'climate_zone' in df_test.columns}")
print(f"df_test zones: {df_test['climate_zone'].value_counts().to_dict()}")

# Stratified XAI sample — 200 per zone
xai_frames = []
for zone in df_test['climate_zone'].dropna().unique():
    zone_df = df_test[df_test['climate_zone'] == zone]
    n       = min(len(zone_df), 200)
    xai_frames.append(zone_df.sample(n=n, random_state=42))

xai_df = pd.concat(xai_frames).reset_index(drop=True)
X_samp = xai_df[feature_cols].reset_index(drop=True)
y_samp = xai_df['total_kwh'].reset_index(drop=True)

print(f"climate_zone in xai_df: {'climate_zone' in xai_df.columns}")
print(f"xai_df zones: {xai_df['climate_zone'].value_counts().to_dict()}")
print(f"XAI sample: {xai_df.shape}")
print(f"Features: {len(feature_cols)}")

# ── SHAP ──────────────────────────────────────────────────────
print('\nComputing SHAP...')
explainer   = shap.TreeExplainer(gb_model)
shap_values = explainer.shap_values(X_samp)
shap_imp = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_imp.to_csv(RESULTS/'xai'/'shap'/'global_importance.csv', index=False)

print('Top 5 SHAP features:')
for _, r in shap_imp.head(5).iterrows():
    print(f'  {r["feature"]:<35} {r["mean_abs_shap"]:.4f}')

# Per-zone SHAP heatmap
zone_shap = {}
for zone, grp in xai_df.groupby('climate_zone'):
    mask = xai_df['climate_zone'].values == zone
    zone_shap[zone] = np.abs(shap_values[mask]).mean(axis=0)
pd.DataFrame(zone_shap, index=feature_cols).T\
    .to_csv(RESULTS/'xai'/'shap'/'zone_shap_importance.csv')

# SHAP global plot
fig, ax = plt.subplots(figsize=(10,7))
top15 = shap_imp.head(15)
ax.barh(range(len(top15)), top15['mean_abs_shap'].values[::-1],
        color='#1D9E75', edgecolor='white')
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['feature'].values[::-1], fontsize=10)
ax.set_xlabel('Mean |SHAP| (kWh)')
ax.set_title(f'Global Feature Importance — SHAP ({RUN_MODE})', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS/'xai'/'plots'/'shap_global.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ SHAP complete')

# ── LIME ──────────────────────────────────────────────────────
print('\nComputing LIME...')
lime_exp = LimeTabularExplainer(
    X_test.values, feature_names=feature_cols,
    mode='regression', random_state=42, discretize_continuous=True)
lime_results = []
for zone, grp in xai_df.groupby('climate_zone'):
    grp = grp.reset_index(drop=True)
    for label, row_i in [('low',  int(grp['total_kwh'].idxmin())),
                          ('mid',  len(grp)//2),
                          ('high', int(grp['total_kwh'].idxmax()))]:
        inst = X_samp.iloc[row_i].values
        try:
            exp = lime_exp.explain_instance(
                inst, gb_model.predict, num_features=8, num_samples=500)
            rec = {'zone': zone, 'level': label,
                   'actual_kwh':    float(grp.loc[row_i,'total_kwh']),
                   'predicted_kwh': float(gb_model.predict(inst.reshape(1,-1))[0])}
            for j, (rule, impact) in enumerate(exp.as_list()[:5], 1):
                rec[f'rule_{j}']   = rule
                rec[f'impact_{j}'] = round(float(impact), 4)
            lime_results.append(rec)
        except Exception as e:
            print(f'    {zone} {label} failed: {str(e)[:50]}')
    print(f'  Zone {zone}: ✓')
pd.DataFrame(lime_results).to_csv(
    RESULTS/'xai'/'lime'/'lime_explanations.csv', index=False)
print('✓ LIME complete')

# ── Anchors ───────────────────────────────────────────────────
print('\nComputing Anchors...')
y_pred_all    = gb_model.predict(X_test)
low_t, high_t = np.percentile(y_pred_all, 33), np.percentile(y_pred_all, 67)

def predict_class(X):
    p = gb_model.predict(X)
    return np.where(p <= low_t, 0, np.where(p <= high_t, 1, 2))

try:
    from anchor import anchor_tabular
    anch = anchor_tabular.AnchorTabularExplainer(
        ['low','mid','high'], feature_cols,
        X_test.values, discretizer='quartile')
    anchor_results = []
    for zone, grp in xai_df.groupby('climate_zone'):
        hi   = int(grp['total_kwh'].idxmax())
        inst = X_samp.iloc[hi].values
        try:
            exp  = anch.explain_instance(
                inst, predict_class,
                threshold=0.95, delta=0.1, beam_size=2, max_anchor_size=4)
            rule = ' AND '.join(exp.names())
            print(f'  Zone {zone}: IF {rule} (precision={exp.precision():.0%})')
            anchor_results.append({
                'zone': zone, 'rule': rule,
                'precision': round(exp.precision(), 3),
                'coverage':  round(exp.coverage(), 3)
            })
        except Exception as e:
            print(f'  Zone {zone}: failed — {str(e)[:60]}')
    pd.DataFrame(anchor_results).to_csv(
        RESULTS/'xai'/'anchors'/'anchor_rules.csv', index=False)
    print('✓ Anchors complete')
except ImportError:
    print('  anchor-exp not installed — pip install anchor-exp')

print(f'\n✓ All XAI outputs saved to {RESULTS}/xai/')

1yr GB   mean total_kwh: 2.4464
2yr Prophet mean total_kwh: 2.4464

Same building 104523 at 2018-10-01 00:00:00:
  1yr GB total_kwh   : [0.546]
  2yr Prophet total_kwh: [0.546]


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 13: HYBRID ENSEMBLE
# 2-year Prophet + 1-year GB
# Both use release_1.1 — total_kwh values confirmed identical
# ════════════════════════════════════════════════════════════

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, mean_squared_log_error)

# ── Paths ─────────────────────────────────────────────────────
DIR_1YR = Path("run1_1year_r2")
DIR_2YR = Path("run2_2year_r2")
DIR_HYB = Path("run_hybrid_r2")

for d in [DIR_HYB/'predictions', DIR_HYB/'results'/'evaluation'/'plots']:
    d.mkdir(parents=True, exist_ok=True)

# ── Load predictions ──────────────────────────────────────────
print("Loading predictions...")
gb_1yr   = pd.read_parquet(DIR_1YR/'predictions'/'gb_predictions.parquet')
p_2yr    = pd.read_parquet(DIR_2YR/'predictions'/'prophet_predictions.parquet')
hw_1yr   = pd.read_parquet(DIR_1YR/'predictions'/'holt_winters_predictions.parquet')

gb_1yr['building_id'] = gb_1yr['building_id'].astype(str)
p_2yr['building_id']  = p_2yr['building_id'].astype(str)

print(f"1yr GB      : {gb_1yr.shape}")
print(f"2yr Prophet : {p_2yr.shape}")

# ── Merge on building_id + timestamp ─────────────────────────
df = gb_1yr[['building_id','timestamp','total_kwh',
             'climate_zone','gb_prediction']].merge(
    p_2yr[['building_id','timestamp','prophet_prediction',
           'prophet_trend','prophet_residual']],
    on=['building_id','timestamp'], how='inner')

print(f"\nMerged : {df.shape}")
print(f"Buildings : {df['building_id'].nunique()}")
print(f"Timestamps: {df['timestamp'].min()} → {df['timestamp'].max()}")

# Confirm ground truth consistency
print(f"\nGround truth check:")
print(f"  1yr GB mean total_kwh   : {gb_1yr['total_kwh'].mean():.4f}")
print(f"  2yr Prophet mean total_kwh: {p_2yr['total_kwh'].mean():.4f}")
print(f"  Merged mean total_kwh   : {df['total_kwh'].mean():.4f}  ✓")

# ── Metrics helper ────────────────────────────────────────────
def metrics(y_true, y_pred, label=''):
    clean = pd.Series(y_true).reset_index(drop=True) > 0
    yt    = pd.Series(y_true).reset_index(drop=True)[clean]
    yp    = pd.Series(y_pred).reset_index(drop=True)[clean].clip(lower=0.001)
    mape_m = yt > 0.5
    return {
        'model': label,
        'mae'  : mean_absolute_error(yt, yp),
        'rmse' : np.sqrt(mean_squared_error(yt, yp)),
        'rmsle': np.sqrt(mean_squared_log_error(yt, yp)),
        'r2'   : r2_score(yt, yp),
        'mape' : np.mean(np.abs((yt[mape_m]-yp[mape_m])/yt[mape_m]))*100
    }

# ── Grid search optimal weights per zone ─────────────────────
print("\n" + "-"*60)
print("GRID SEARCH — Optimal Prophet weight per zone")
print("-"*60)

clean_df     = df[df['total_kwh'] > 0].copy()
best_weights = {}

for zone in sorted(df['climate_zone'].dropna().unique()):
    z = clean_df[clean_df['climate_zone'] == zone]
    best_r, best_w = 999, 0.0
    for w_p in np.arange(0.0, 0.51, 0.025):
        pred = (w_p * z['prophet_prediction'] +
                (1-w_p) * z['gb_prediction']).clip(lower=0.001)
        r = np.sqrt(mean_squared_log_error(z['total_kwh'], pred))
        if r < best_r:
            best_r, best_w = r, w_p
    best_weights[zone] = {'prophet': best_w, 'gb': round(1-best_w, 3)}

    # Also show GB-alone RMSLE for comparison
    gb_r = np.sqrt(mean_squared_log_error(
        z['total_kwh'], z['gb_prediction'].clip(lower=0.001)))
    diff = best_r - gb_r
    print(f"  Zone {zone}: P={best_w:.3f} GB={1-best_w:.3f}  "
          f"RMSLE={best_r:.4f}  vs GB alone={gb_r:.4f}  ({diff:+.4f})")

# ── Apply optimal weights ─────────────────────────────────────
df['w_p']  = df['climate_zone'].apply(
    lambda z: best_weights.get(z, {'prophet':0.0})['prophet'])
df['w_gb'] = df['climate_zone'].apply(
    lambda z: best_weights.get(z, {'gb':1.0})['gb'])
df['hybrid_prediction'] = (
    df['w_p'] * df['prophet_prediction'] +
    df['w_gb'] * df['gb_prediction']
).clip(lower=0)
df['hybrid_residual'] = df['total_kwh'] - df['hybrid_prediction']

# ── Compute all metrics ───────────────────────────────────────
m_gb     = metrics(df['total_kwh'], df['gb_prediction'],      'GB (1yr)')
m_p      = metrics(df['total_kwh'], df['prophet_prediction'], 'Prophet (2yr)')
m_hybrid = metrics(df['total_kwh'], df['hybrid_prediction'],  'Hybrid (2yr P + 1yr GB)')
m_hw     = metrics(hw_1yr['total_kwh'], hw_1yr['hw_prediction'], 'Holt-Winters (1yr)')

# ── Print comparison ──────────────────────────────────────────
print("\n" + "="*68)
print("HYBRID ENSEMBLE RESULTS")
print("="*68)

prev = [
    ('Persistence',    0.2434, 0.712),
    ('Seasonal Naive', 0.3027, 0.553),
]
print(f"\n{'Model':<30} {'RMSLE':>8} {'R²':>8} {'MAE':>8}")
print("-"*58)
for name, rmsle, r2 in prev:
    print(f"  {name:<28} {rmsle:>8.4f} {r2:>8.4f}  [baseline]")
for m in [m_hw, m_p, m_gb, m_hybrid]:
    marker = "  ← FINAL" if 'Hybrid' in m['model'] else ""
    print(f"  {m['model']:<28} {m['rmsle']:>8.4f} {m['r2']:>8.4f} "
          f"{m['mae']:>8.4f}{marker}")

# RMSLE improvement of hybrid over baselines
print(f"\nRMSLE improvements of Hybrid over baselines:")
for name, rmsle, _ in prev:
    imp = (rmsle - m_hybrid['rmsle']) / rmsle * 100
    print(f"  vs {name:<18}: {imp:>6.1f}%  {'✓' if imp>=15 else '✗'}")
for m in [m_hw, m_p]:
    imp = (m['rmsle'] - m_hybrid['rmsle']) / m['rmsle'] * 100
    print(f"  vs {m['model']:<18}: {imp:>6.1f}%  {'✓' if imp>=15 else '✗'}")

# Compare hybrid vs both standalone models
print(f"\nHybrid vs standalone comparison:")
print(f"  1yr GB alone    : RMSLE={m_gb['rmsle']:.4f}  R²={m_gb['r2']:.4f}")
print(f"  2yr Ensemble    : RMSLE=0.1879  R²=0.868  [from run2]")
print(f"  Hybrid (2P+1GB) : RMSLE={m_hybrid['rmsle']:.4f}  R²={m_hybrid['r2']:.4f}")

# Daily aggregate
print(f"\nDAILY AGGREGATE:")
df['date'] = pd.to_datetime(df['timestamp']).dt.date
for pred_col, label in [('gb_prediction','GB (1yr)'),
                         ('prophet_prediction','Prophet (2yr)'),
                         ('hybrid_prediction','Hybrid')]:
    daily = df.groupby(['building_id','date']).agg(
        total_kwh=('total_kwh','sum'),
        prediction=(pred_col,'sum')
    ).reset_index()
    dc = daily[daily['total_kwh']>0].copy()
    dc['prediction'] = dc['prediction'].clip(lower=0.001)
    rmsle = np.sqrt(mean_squared_log_error(dc['total_kwh'], dc['prediction']))
    r2    = r2_score(dc['total_kwh'], dc['prediction'])
    print(f"  {label:<22} RMSLE={rmsle:.4f}  R²={r2:.4f}")

# Per zone
print(f"\nPER ZONE — Hybrid:")
for zone in sorted(df['climate_zone'].dropna().unique()):
    z  = df[df['climate_zone']==zone]
    zm = metrics(z['total_kwh'], z['hybrid_prediction'], zone)
    w  = best_weights.get(zone, {'prophet':0,'gb':1})
    gb_m = metrics(z['total_kwh'], z['gb_prediction'], '')
    diff = zm['rmsle'] - gb_m['rmsle']
    print(f"  Zone {zone}: RMSLE={zm['rmsle']:.4f}  R²={zm['r2']:.4f}  "
          f"P={w['prophet']:.3f}  diff vs 1yr GB: {diff:+.4f}")

# ── Save ──────────────────────────────────────────────────────
print("\n" + "-"*60)
print("SAVING")
print("-"*60)

df.to_parquet(DIR_HYB/'predictions'/'hybrid_predictions.parquet', index=False)
print(f"✓ Predictions : {DIR_HYB}/predictions/hybrid_predictions.parquet")

pd.DataFrame([m_hw, m_p, m_gb, m_hybrid]).to_csv(
    DIR_HYB/'results'/'metrics_summary.csv', index=False)
print(f"✓ Metrics     : {DIR_HYB}/results/metrics_summary.csv")

pd.DataFrame([{'zone':z,**w} for z,w in best_weights.items()]).to_csv(
    DIR_HYB/'results'/'ensemble_weights.csv', index=False)
print(f"✓ Weights     : {DIR_HYB}/results/ensemble_weights.csv")

print(f"\n{'='*60}")
print("HYBRID ENSEMBLE COMPLETE")
print(f"{'='*60}")
print(f"  2yr Prophet RMSLE : {m_p['rmsle']:.4f}")
print(f"  1yr GB RMSLE      : {m_gb['rmsle']:.4f}")
print(f"  Hybrid RMSLE      : {m_hybrid['rmsle']:.4f}")
beat_gb = m_hybrid['rmsle'] < m_gb['rmsle']
print(f"  Beats 1yr GB      : {'✓ YES' if beat_gb else '✗ NO'}")
beat_2yr = m_hybrid['rmsle'] < 0.1879
print(f"  Beats 2yr Ensemble: {'✓ YES' if beat_2yr else '✗ NO'}")

Loading predictions...
1yr GB      : (220900, 75)
2yr Prophet : (220900, 26)

Merged : (220900, 8)
Buildings : 100
Timestamps: 2018-10-01 00:00:00 → 2019-01-01 00:00:00

Ground truth check:
  1yr GB mean total_kwh   : 2.4464
  2yr Prophet mean total_kwh: 2.4464
  Merged mean total_kwh   : 2.4464  ✓

------------------------------------------------------------
GRID SEARCH — Optimal Prophet weight per zone
------------------------------------------------------------
  Zone 2B: P=0.075 GB=0.925  RMSLE=0.1662  vs GB alone=0.1677  (-0.0015)
  Zone 3B: P=0.050 GB=0.950  RMSLE=0.1792  vs GB alone=0.1801  (-0.0008)
  Zone 3C: P=0.075 GB=0.925  RMSLE=0.2042  vs GB alone=0.2053  (-0.0012)
  Zone 4B: P=0.025 GB=0.975  RMSLE=0.2078  vs GB alone=0.2079  (-0.0001)
  Zone 4C: P=0.075 GB=0.925  RMSLE=0.1786  vs GB alone=0.1797  (-0.0011)
  Zone 5B: P=0.100 GB=0.900  RMSLE=0.1888  vs GB alone=0.1904  (-0.0017)

HYBRID ENSEMBLE RESULTS

Model                             RMSLE       R²      MAE
---------

In [58]:
import pandas as pd
from pathlib import Path

# Run this as a quick diagnostic cell
print(f'PROCESSED = {PROCESSED}')
df_check = pd.read_parquet(PROCESSED / 'engineered_features.parquet')
print(f'Parquet shape: {df_check.shape}')
print(f'Has kwh_lag_24: {"kwh_lag_24" in df_check.columns}')
print(f'Has temp_x_hour_sin: {"temp_x_hour_sin" in df_check.columns}')
print(f'Has floor_x_cdh: {"floor_x_cdh" in df_check.columns}')

PROCESSED = run2_2year_r2/processed
Parquet shape: (1754400, 108)
Has kwh_lag_24: True
Has temp_x_hour_sin: True
Has floor_x_cdh: True


In [59]:
exc = ['building_id','timestamp','total_kwh','gisjoin','climate_zone','data_year',
       'floor_area_num','vintage_num','vintage_age',
       'hour','day_of_week','month','week_of_year',
       'kwh_delta_24h','kwh_delta_168h',
       ]

df_check = pd.read_parquet(PROCESSED / 'engineered_features.parquet')
feature_cols = [c for c in df_check.columns if c not in exc]

print(f'Total columns     : {df_check.shape[1]}')
print(f'Excluded columns  : {len(exc)}')
print(f'Feature cols      : {len(feature_cols)}')
print(f'\nExcluded list:')
for c in exc:
    present = c in df_check.columns
    print(f'  {c:<30} in parquet: {present}')

Total columns     : 108
Excluded columns  : 15
Feature cols      : 93

Excluded list:
  building_id                    in parquet: True
  timestamp                      in parquet: True
  total_kwh                      in parquet: True
  gisjoin                        in parquet: True
  climate_zone                   in parquet: True
  data_year                      in parquet: True
  floor_area_num                 in parquet: True
  vintage_num                    in parquet: True
  vintage_age                    in parquet: True
  hour                           in parquet: True
  day_of_week                    in parquet: True
  month                          in parquet: True
  week_of_year                   in parquet: True
  kwh_delta_24h                  in parquet: True
  kwh_delta_168h                 in parquet: True


In [73]:
# Verify no test timestamps appear in training set
train_ts = set(zip(df_train['building_id'].astype(str), 
                   df_train['timestamp'].astype(str)))
test_ts  = set(zip(df_test['building_id'].astype(str),  
                   df_test['timestamp'].astype(str)))

overlap = train_ts & test_ts
print(f'Train-test overlap: {len(overlap)} rows')
print(f'Expected          : 0 rows')

Train-test overlap: 0 rows
Expected          : 0 rows
